### QLLVM优化性能测试

#### 0. 全面性能基准（推荐）

对 **QLLVM、Qiskit、Cirq、PennyLane** 在同一批 OpenQASM 用例上统计：

| 指标 | 说明 |
|------|------|
| **编译后门数** | 编译后线路的 `QuantumCircuit.size()`（门操作数） |
| **线路深度** | `QuantumCircuit.depth()` |
| **编译时间** | 单次编译耗时（秒） |
| **内存开销** | Python 后端：`tracemalloc` 峰值；QLLVM 子进程：采样 RSS 峰值（近似） |

**说明**：门数/深度统一用 Qiskit 解析编译后的 QASM，便于横向对比；Cirq 使用 `CZTargetGateset` 优化，PennyLane 使用 `qml.compile`（RZ/RX/RY/H/CNOT），与 Qiskit/QLLVM 的 `{rx,ry,rz,cz,h}` 基组不完全相同，数值仅作趋势参考。

**依赖**（conda `qllvm` 环境）：`cirq-core`、`ply`（Cirq 读 QASM）、`pennylane`、`psutil`、`matplotlib`、`pandas`。若缺失可执行：`pip install cirq-core ply pennylane psutil matplotlib pandas`。

**论文用数据与图表（推荐）**：在 `test/` 下执行  
`python run_paper_benchmark.py --out paper_benchmark_out_full`  
可**断点续跑**（中断后去掉 `--no-resume` 再运行即可）。产出：`benchmark_long.csv`（长表）、`benchmark_summary.csv`（汇总）、`fig1`–`fig4` PNG 图。详见 `run_paper_benchmark.py` 顶部说明。

In [ ]:
import os
import pandas as pd

from utils.get_qasm_file import get_qasm_files
from utils.benchmark_compilers import benchmark_one_file, results_to_rows

# 试跑时限制文件数；全量测试改为 MAX_FILES = None
MAX_FILES = 5

_candidate = os.path.expanduser("~/.qllvm/bin/qllvm")
qllvm_bin = _candidate if os.path.isfile(_candidate) else "qllvm"

qasm_folder = "./MQTBench"
qasm_files = get_qasm_files(qasm_folder)
if MAX_FILES is not None:
    qasm_files = qasm_files[:MAX_FILES]

rows_all = []
for i, p in enumerate(qasm_files):
    print(f"[{i + 1}/{len(qasm_files)}] {os.path.basename(p)}")
    rows_all.extend(
        results_to_rows(
            benchmark_one_file(
                p,
                backends=["qllvm", "qiskit", "cirq", "pennylane"],
                qllvm_bin=qllvm_bin,
            )
        )
    )

df = pd.DataFrame(rows_all)
df["file"] = df["qasm_path"].apply(os.path.basename)

err_df = df[df["error"].notna()]
if len(err_df):
    print("部分后端报错（前几条）：")
    display(err_df[["file", "backend", "error"]].head(20))

summary = (
    df.groupby("backend", dropna=False)
    .agg(
        gate_count_mean=("gate_count", "mean"),
        depth_mean=("depth", "mean"),
        compile_time_s_mean=("compile_time_s", "mean"),
        memory_peak_MiB_mean=("memory_peak_MiB", "mean"),
    )
    .round(4)
)
print("按后端汇总（均值）：")
display(summary)

wide = df.pivot_table(
    index="file",
    columns="backend",
    values=["gate_count", "depth", "compile_time_s", "memory_peak_MiB"],
    aggfunc="first",
)
print("逐文件对比：")
display(wide)

以下为**分步导出各编译器 qasm** 的旧流程（仍含 Qpanda）；**QLLVM 与 Qiskit、Cirq、PennyLane 的统一门数/深度/时间/内存对比请见上一节「0. 全面性能基准」**。

#### 1. 使用QLLVM依次编译测试集中的各测试用例，并获取编译后的qasm文件

In [ ]:
from utils.get_qasm_file import get_qasm_files
import os
qasm_folder = "./MQTBench" 
qasm_files = get_qasm_files(qasm_folder)

for i, qasm_file in enumerate(qasm_files):
    algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
    compile_command = f"qllvm {qasm_file} -qrt nisq -qpu qasm-backend -o ./QLLVM/{algorithm_name} -O1 -basicgate=[rx,ry,rz,cz,h]"
    print(f"i: {i} {compile_command}")
    os.system(compile_command)

#### 2.使用Qiskit依次编译测试集中的各测试用例，并获取编译后的qasm文件

In [ ]:
import qiskit
from qiskit import QuantumCircuit,transpile
from utils.get_qasm_file import get_qasm_files
from qiskit_aer import Aer
from qiskit.qasm2 import dumps
import os

qasm_folder = "./MQTBench" 
qasm_files = get_qasm_files(qasm_folder)
for i, qasm_file in enumerate(qasm_files):
    algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
    print(f"{qasm_file}")
    qc = qiskit.QuantumCircuit().from_qasm_file(qasm_file)
    transpiled_circuit = transpile(qc,basis_gates=['rx','ry','rz','cz','h'],optimization_level=3)
    qasm_str = dumps(transpiled_circuit)
    compiled_path = f"./Qiskit/{algorithm_name}.qasm"
    with open(compiled_path, "w") as f:
        f.write(qasm_str)

#### 3. 使用Qpanda依次编译测试集中的各测试用例，并获取编译后的qasm文件

In [ ]:
from pyqpanda3.core import *
from pyqpanda3.transpilation import Transpiler
from pyqpanda3.intermediate_compiler.intermediate_compiler import convert_qasm_file_to_qprog,convert_qprog_to_qasm
from utils.topo import generate_fully_connected_topology
from utils.get_qasm_file import get_qasm_files
import os

qasm_folder = "./MQTBench" 
qasm_files = get_qasm_files(qasm_folder)
topo = generate_fully_connected_topology(50)

for i, qasm_file in enumerate(qasm_files):
    print(f"{qasm_file}")
    algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
    qprog = convert_qasm_file_to_qprog(qasm_file)
    transpiler = Transpiler()
    basic_gates = ['RX','RY','RZ','CZ','H']
    prog_level_3 = transpiler.transpile(qprog, topo, {}, 2,basic_gates)
    qasm_str = convert_qprog_to_qasm(prog_level_3)
    compiled_path = f"./Qpanda/{algorithm_name}.qasm"
    with open(compiled_path, "w") as f:
        f.write(qasm_str)

#### 4.使用PennyLane依次编译测试集中的各测试用例，并获取编译后的qasm文件

In [ ]:
import re
import pennylane as qml
from utils.get_qasm_file import get_qasm_files
import os

def qasm_qubit_count(qasm_str: str) -> int:
    """OpenQASM 2：对所有 qreg 的长度求和，得到线路总比特数。"""
    return sum(int(m.group(1)) for m in re.finditer(r"qreg\s+\w+\[(\d+)\]", qasm_str))


qasm_folder = "./MQTBench"
qasm_files = get_qasm_files(qasm_folder)

for i, qasm_file in enumerate(qasm_files):
    print(f"{qasm_file}")
    algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
    with open(qasm_file, encoding="utf-8") as f:
        qasm_str = f.read()
    n = qasm_qubit_count(qasm_str)
    fn = qml.from_qasm(qasm_str, measurements=qml.probs(wires=range(n)))
    fn_compiled = qml.compile(
        fn,
        basis_set=["RZ", "RX", "RY", "H", "CNOT"],
        num_passes=2
    )
    dev = qml.device("default.qubit", wires=n)
    qnode = qml.QNode(fn_compiled, dev)
    qasm_out = qml.to_openqasm(qnode, measure_all=False, rotations=False)()
    os.makedirs("./Pennylane", exist_ok=True)
    compiled_path = f"./Pennylane/{algorithm_name}.qasm"
    with open(compiled_path, "w") as f:
        f.write(qasm_out)

#### 5. 对比不同编译器的优化后qasm文件的门个数、深度变化，计算QLLVM相比于Qiskit、QPanda与PennyLane的门个数优化百分比、线路深度优化百分比

（1）QLLVM与Qiskit对比

In [ ]:
from utils.compare_data import compare_qasm_diff
from utils.show_data import show_CN
qiskit_compiled_path = "./Qiskit"
qllvm_compiled_path = "./QLLVM" 
output_csv = "./result/qllvm_qiskit.csv"
compare_qasm_diff(qiskit_compiled_path,qllvm_compiled_path,output_csv)
show_CN(output_csv)

（2）QLLVM与Qpanda对比

In [ ]:
from utils.compare_data import compare_qasm_diff
from utils.show_data import show_CN
qpanda_compiled_path = "./Qpanda"
qllvm_compiled_path = "./QLLVM"
output_csv = "./result/qllvm_qpanda.csv"
compare_qasm_diff(qpanda_compiled_path,qllvm_compiled_path,output_csv)
show_CN(output_csv)

（3）QLLVM与PennyLane对比

In [ ]:
from utils.compare_data import compare_qasm_diff
from utils.show_data import show_CN
pennylane_compiled_path = "./Pennylane"
qllvm_compiled_path = "./QLLVM"
output_csv = "./result/qllvm_pennylane.csv"
compare_qasm_diff(pennylane_compiled_path,qllvm_compiled_path,output_csv)
show_CN(output_csv)